In [1]:
using Lux, LuxCUDA, MLUtils
using Optimisers, Random, Statistics
using Zygote
using DiffEqFlux, OrdinaryDiffEq
using FFTW
#using Reactant
using Images, JLD2
using ComponentArrays
using DeepEquilibriumNetworks
using Plots
using Dates
using SciMLSensitivity, Lux, NonlinearSolve, LinearSolve


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; 

In [2]:
#Reactant.set_default_backend("cpu")
#const device = reactant_device()
const cdev = cpu_device()
const gdev = gpu_device()
dev = gdev

(::CUDADevice{Nothing, Missing}) (generic function with 1 method)

In [3]:
model_first_conv = Chain(
        
        Conv((5, 5), 1 => 8, tanh; pad = 2, init_weight=truncated_normal(; std=0.01)),
        Conv((5, 5), 8 => 16; pad = 2,init_weight=truncated_normal(; std=0.01)),
        SkipConnection(Chain(
            Conv((1, 1), 16 => 64, gelu,init_weight=truncated_normal(; std=0.01)),
            Conv((1, 1), 64 => 32, gelu,init_weight=truncated_normal(; std=0.01)),
            Conv((1, 1), 32 => 16,init_weight=truncated_normal(; std=0.01)),
        ), +),
    ) 
model_conv = Chain(
        Parallel(+, 
            NoOpLayer(), # Pass z through
            NoOpLayer()  # Pass x through
        ),
        model_first_conv,
        GroupNorm(16, 4),
        Conv((5, 5), 16 => 32, tanh; pad = 2,init_weight=truncated_normal(; std=0.01)),
        Conv((5, 5), 32 => 64; pad = 2,init_weight=truncated_normal(; std=0.01)),
        SkipConnection(Chain(
            Conv((1, 1), 64 => 128, gelu,init_weight=truncated_normal(; std=0.01)),
            Conv((1, 1), 128 => 128, gelu,init_weight=truncated_normal(; std=0.01)),
            Conv((1, 1), 128 => 64, gelu,init_weight=truncated_normal(; std=0.01)),
        ), +),
        GroupNorm(64, 4),
        Conv((1, 1), 64 => 64, gelu,init_weight=truncated_normal(; std=0.01)),
        Conv((1, 1), 64 => 32, gelu,init_weight=truncated_normal(; std=0.01)),
        Conv((1, 1), 32 => 1,init_weight=truncated_normal(; std=0.01)),
    ) 


Chain(
    layer_1 = Parallel(
        connection = +,
        layer_(1-2) = NoOpLayer(),
    ),
    layer_2 = Chain(
        layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
        layer_2 = Conv((5, 5), 8 => 16, pad=2),   # 3_216 parameters
        layer_3 = SkipConnection(
            connection = +,
            layers = Chain(
                layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
            ),
        ),
    ),
    layer_3 = GroupNorm(16, 4, affine=true),      # 32 parameters
    layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
    layer_5 = Conv((5, 5), 32 => 64, pad=2),      # 51_264 parameters
    layer_6 = SkipConnection(
        connection = +,
        layers = Chain(
            layer_1 = Conv((1, 1), 64 => 128, gelu_tanh),  # 8_320 parameters
            layer_2 = Con

In [4]:
model = SkipConnection(connection = +, layers = DeepEquilibriumNetwork(model_conv, NewtonRaphson(; linsolve=KrylovJL_GMRES()); init = nothing, verbose=false, linsolve_kwargs=(; maxiters=10), maxiters=10))

SkipConnection(
    connection = +,
    layers = DeepEquilibriumNetwork(
        model = Chain(
            layer_1 = Parallel(
                connection = +,
                layer_(1-2) = NoOpLayer(),
            ),
            layer_2 = Chain(
                layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
                layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                layer_3 = SkipConnection(
                    connection = +,
                    layers = Chain(
                        layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                        layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                        layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                    ),
                ),
            ),
            layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
            layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
            layer_5 = C

In [ ]:
#model = DeepEquilibriumNetwork(model_conv, Tsit5(); init = nothing,save_everystep = false, reltol = 1e-3, abstol = 1e-4, save_start = false)

DeepEquilibriumNetwork(
    model = Chain(
        layer_1 = Parallel(
            connection = +,
            layer_(1-2) = NoOpLayer(),
        ),
        layer_2 = Chain(
            layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
            layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
            layer_3 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                    layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                    layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                ),
            ),
        ),
        layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
        layer_4 = Conv((5, 5), 16 => 32, tanh, pad=2),  # 12_832 parameters
        layer_5 = Conv((5, 5), 32 => 64, pad=2),  # 51_264 parameters
        layer_6 = SkipConnection(
            connection = +,
            laye

In [5]:
rng = Xoshiro()
ps , st = Lux.setup(rng, model)

((model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = (weight = Float32[-0.0011323299 0.0024668938 … -0.005750823 -0.009401493; 0.0037383484 -0.0018955966 … -0.010178952 0.0041503003; … ; 0.007482292 -0.006937391 … 0.0017269709 -0.0074712303; -0.019171625 0.0035387247 … -0.016605204 0.0116598215;;;; 0.0068981745 0.0019738122 … -0.00606917 -0.005334116; -0.017366316 -0.014927602 … 0.0017087468 -0.0142215025; … ; -0.0008590457 0.00013170131 … 0.0015930333 -0.004310344; 0.011666445 -0.00964382 … -0.008574876 -0.0008333815;;;; 0.021645036 -0.008688027 … 0.008393177 0.009223258; 0.01039143 -0.0005005241 … -0.0037463105 -0.01807476; … ; -0.014507019 0.0057819304 … -0.009816878 -0.0007498646; -0.011406054 0.014663853 … -0.007332031 0.013991792;;;; 0.010460159 -0.0011300324 … 0.0047192313 0.001177789; -0.012844865 0.00932131 … -0.0033571513 0.0007395658; … ; 0.0067352382 -0.01323358 … -0.010928341 0.00748152; -0.010096995 0.012415351 … 0.0010848972 0.01667

In [7]:
@generated function copy_subtree!(A, B)
    fields = fieldnames(B)
    quote
        Base.@_inline_meta
        $(Expr(:block, [
            :(if hasfield(typeof(A), $(QuoteNode(f)))
                a_val = getfield(A, $(QuoteNode(f)))
                b_val = getfield(B, $(QuoteNode(f)))
                if isa(b_val, AbstractArray)
                    a_val .= b_val
                else
                    copy_subtree!(a_val, b_val)
                end
            end)
        for f in fields]...))
        A
    end
end

copy_subtree! (generic function with 1 method)

In [9]:
copy_subtree!(ps,ps_old)

(model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = (weight = Float32[-0.44732723 0.11423418 … 0.5138864 0.51249886; -0.5446564 0.5458368 … -0.48519784 0.093109705; … ; -0.44819477 -0.5135532 … -0.053549346 0.516115; 0.37991655 0.21562752 … 0.38979274 -0.53646654;;;; -0.44932598 -0.2054805 … 0.21603034 0.42200413; -0.39136347 -0.26383945 … -0.4626396 -0.09395185; … ; -0.06426672 0.535148 … -0.33758798 0.56995434; 0.21464957 0.49491113 … -0.08579486 0.40112796;;;; 0.5098958 -0.5186666 … 0.19954678 -0.1553303; 0.011289931 -0.5516167 … 0.049989898 0.110431366; … ; 0.51556593 0.52508384 … 0.51935464 -0.07630795; -0.18180521 0.558303 … -0.08433672 -0.11575433;;;; 0.07098877 0.17111784 … 0.44928318 -0.1569036; -0.32379273 -0.5118967 … -0.3868064 0.46889165; … ; 0.41754147 -0.056241043 … -0.056377664 -0.20746832; -0.20662466 -0.11932651 … -0.024522118 0.49880424;;;; 0.41276863 -0.29503915 … -0.5351137 -0.3661332; -0.0006586602 0.19536184 … 0.19633587 0.1

In [16]:
function print_fieldtree(x; indent=0)
    T = typeof(x)
    for name in fieldnames(T)
        val = getfield(x, name)
        println(" "^indent * string(name), " :: ", typeof(val))
        if isstructtype(typeof(val)) && !(val isa AbstractArray)
            print_fieldtree(val; indent=indent+2)
        end
    end
end

print_fieldtree (generic function with 1 method)

In [17]:
print_fieldtree(ps)

model :: @NamedTuple{layer_1::@NamedTuple{layer_1::@NamedTuple{}, layer_2::@NamedTuple{}}, layer_2::@NamedTuple{layer_1::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_2::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_3::@NamedTuple{layer_1::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_2::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_3::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}}}, layer_3::@NamedTuple{scale::Vector{Float32}, bias::Vector{Float32}}, layer_4::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_5::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_6::@NamedTuple{layer_1::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_2::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}, layer_3::@NamedTuple{weight::Array{Float32, 4}, bias::Vector{Float32}}}, layer_7::@NamedTuple{scale::Vector{Float32}, bias::Vec

In [20]:
print_fieldtree(ps_old)

data :: CuArray{Float32, 1, CUDA.DeviceMemory}
axes :: Tuple{ComponentArrays.Axis{(layer_1 = ViewAxis(1:7120, Axis(layer_1 = ViewAxis(1:208, Axis(weight = ViewAxis(1:200, ShapedAxis((5, 5, 1, 8))), bias = ViewAxis(201:208, Shaped1DAxis((8,))))), layer_2 = ViewAxis(209:3424, Axis(weight = ViewAxis(1:3200, ShapedAxis((5, 5, 8, 16))), bias = ViewAxis(3201:3216, Shaped1DAxis((16,))))), layer_3 = ViewAxis(3425:7120, Axis(layer_1 = ViewAxis(1:1088, Axis(weight = ViewAxis(1:1024, ShapedAxis((1, 1, 16, 64))), bias = ViewAxis(1025:1088, Shaped1DAxis((64,))))), layer_2 = ViewAxis(1089:3168, Axis(weight = ViewAxis(1:2048, ShapedAxis((1, 1, 64, 32))), bias = ViewAxis(2049:2080, Shaped1DAxis((32,))))), layer_3 = ViewAxis(3169:3696, Axis(weight = ViewAxis(1:512, ShapedAxis((1, 1, 32, 16))), bias = ViewAxis(513:528, Shaped1DAxis((16,))))))))), layer_2 = ViewAxis(7121:7152, Axis(scale = ViewAxis(1:16, Shaped1DAxis((16,))), bias = ViewAxis(17:32, Shaped1DAxis((16,))))), layer_3 = ViewAxis(7153:19984, A

In [13]:
ps_old.layer_1

ComponentVector{Float32,SubArray...}(layer_1 = (weight = Float32[-0.5156039 0.31830207 … 0.54628533 0.20379029; -0.054568622 -0.45326352 … 0.42619815 -0.46810475; … ; 0.052183174 -0.10723932 … 0.12594105 -0.31320214; 0.20721574 -0.54464287 … 0.19336234 0.31061116;;;; 0.14788787 -0.005361416 … -0.32622296 0.032514222; -0.18747962 -0.036960877 … 0.20209257 0.25226864; … ; 0.36388606 -0.45006093 … 0.38368112 0.102261476; 0.22638045 -0.045780875 … 0.07437828 -0.099924296;;;; 0.42248276 0.014029021 … -0.112267114 -0.22378059; -0.1426463 -0.3524065 … -0.057274412 -0.54889005; … ; -0.3172466 0.36811435 … -0.4447652 0.5174231; 0.5102624 0.17435746 … -0.43966433 0.14375825;;;; 0.11226511 -0.20781566 … 0.295928 -0.14229618; -0.22354339 -0.10709284 … -0.30813083 -0.55016106; … ; -0.36722964 0.27480987 … 0.30162057 0.3756912; -0.27520102 0.28823578 … 0.51057607 -0.45854425;;;; -0.20631212 0.4666755 … 0.2519986 -0.008068724; 0.37324235 0.14082547 … 0.35090733 0.020776508; … ; -0.46206975 -0.2230401

In [15]:
ps.model.layer_2

(layer_1 = (weight = Float32[-0.44732723 0.11423418 … 0.5138864 0.51249886; -0.5446564 0.5458368 … -0.48519784 0.093109705; … ; -0.44819477 -0.5135532 … -0.053549346 0.516115; 0.37991655 0.21562752 … 0.38979274 -0.53646654;;;; -0.44932598 -0.2054805 … 0.21603034 0.42200413; -0.39136347 -0.26383945 … -0.4626396 -0.09395185; … ; -0.06426672 0.535148 … -0.33758798 0.56995434; 0.21464957 0.49491113 … -0.08579486 0.40112796;;;; 0.5098958 -0.5186666 … 0.19954678 -0.1553303; 0.011289931 -0.5516167 … 0.049989898 0.110431366; … ; 0.51556593 0.52508384 … 0.51935464 -0.07630795; -0.18180521 0.558303 … -0.08433672 -0.11575433;;;; 0.07098877 0.17111784 … 0.44928318 -0.1569036; -0.32379273 -0.5118967 … -0.3868064 0.46889165; … ; 0.41754147 -0.056241043 … -0.056377664 -0.20746832; -0.20662466 -0.11932651 … -0.024522118 0.49880424;;;; 0.41276863 -0.29503915 … -0.5351137 -0.3661332; -0.0006586602 0.19536184 … 0.19633587 0.1968815; … ; -0.45593277 0.5651698 … 0.533905 -0.54658186; 0.11920786 0.4066252 …

In [7]:
ps = ps |> ComponentArray |> dev
st = st |> dev

opt = Optimisers.NAdam(1e-4)
state = Optimisers.setup(opt,ps)
train_state = Lux.Training.TrainState(model, ps, st, opt)

TrainState(
    SkipConnection(
        connection = +,
        layers = DeepEquilibriumNetwork(
            model = Chain(
                layer_1 = Parallel(
                    connection = +,
                    layer_(1-2) = NoOpLayer(),
                ),
                layer_2 = Chain(
                    layer_1 = Conv((5, 5), 1 => 8, tanh, pad=2),  # 208 parameters
                    layer_2 = Conv((5, 5), 8 => 16, pad=2),  # 3_216 parameters
                    layer_3 = SkipConnection(
                        connection = +,
                        layers = Chain(
                            layer_1 = Conv((1, 1), 16 => 64, gelu_tanh),  # 1_088 parameters
                            layer_2 = Conv((1, 1), 64 => 32, gelu_tanh),  # 2_080 parameters
                            layer_3 = Conv((1, 1), 32 => 16),  # 528 parameters
                        ),
                    ),
                ),
                layer_3 = GroupNorm(16, 4, affine=true),  # 32 parameters
       

In [8]:
function loss_function(model, ps, st, (x, y_true))
    y, st = model(x, ps, st)
    y_pred = y
    #y_pred = model(x, ps, st)[1][1]
    loss_mse= MSELoss()
    mse_loss = loss_mse(y_pred, y_true)
    #sptrl_loss = loss_mse(dct(dct(y_pred,1),2),dct(dct(y_pred,1),2))
    return mse_loss, st
    #return mes_loss + sptrl_loss, st
end

loss_function (generic function with 1 method)

In [9]:
x = randn(Float32,128,128,1,4)
y_true = randn(Float32,128,128,1,4)
x_dev = x |> dev
y_dev = y_true |> dev
y, _ = model(x_dev, ps, st)

(Float32[0.06431675 0.118218124 … 1.3552971 -0.43407467; 0.7338029 0.5229141 … 0.009274967 0.7038964; … ; 1.831171 -0.20443156 … -0.7177905 0.5848996; -1.1477447 1.7115551 … -1.1586485 0.29220203;;;; 0.12825942 0.02976447 … 0.5671103 0.3825019; 1.552884 -1.5237911 … -0.7510575 -0.12453018; … ; -0.08008969 0.44822595 … -1.5486904 -1.0909953; 0.37526155 0.33896017 … -1.0976791 -0.1500322;;;; 0.8472634 2.6570952 … -0.048869386 0.76078266; 1.4469411 -1.3207287 … 0.37499532 0.22975995; … ; -0.23338306 -1.090878 … -1.7627325 -0.8995002; 0.025619444 0.5791738 … 0.018899683 -0.23823418;;;; 0.7945188 0.5588832 … -0.72359043 0.4878896; -1.4722091 0.4960555 … -0.24502054 0.48753428; … ; -0.92468524 0.8528082 … -0.20417735 0.6571951; 1.331415 1.9147415 … -0.63144815 -0.21232134], (model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple())), layer_3

In [10]:
loss_function(model,ps,st,(x_dev,y_dev))

(2.0095563f0, (model = (layer_1 = (layer_1 = NamedTuple(), layer_2 = NamedTuple()), layer_2 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple())), layer_3 = NamedTuple(), layer_4 = NamedTuple(), layer_5 = NamedTuple(), layer_6 = (layer_1 = NamedTuple(), layer_2 = NamedTuple(), layer_3 = NamedTuple()), layer_7 = NamedTuple(), layer_8 = NamedTuple(), layer_9 = NamedTuple(), layer_10 = NamedTuple()), fixed_depth = Val{0}(), init = NamedTuple(), solution = DeepEquilibriumSolution
 * Initial Guess: Float32[0.0803871 0.080336 … 0.0803883 0.0804181; 0.0804184 0.0803731 … 0.0803901 0.0803737; … ; 0.0804451 0.0804474 … 0.0804681 0.0804037; 0.080496 0.0805099 … 0.0804569 0.0804035;;;; 0.0803858 0.0803447 … 0.0803864 0.0804152; 0.0804223 0.0803815 … 0.0803878 0.0803729; … ; 0.080448 0.0804521 … 0.080469 0.0804005; 0.0804946 0.0805066 … 0.0804603 0.0804022;;;; 0.0803812 0.0803415 … 0.0803825 0.0804135; 0.0804194 0.080

In [ ]:
     






Training.single_train_step!(AutoZygote(), loss_function, (x_dev, y_dev), train_state)


BoundsError: BoundsError: attempt to access Tuple{Float32, @NamedTuple{model::@NamedTuple{layer_1::@NamedTuple{layer_1::@NamedTuple{}, layer_2::@NamedTuple{}}, layer_2::@NamedTuple{layer_1::@NamedTuple{}, layer_2::@NamedTuple{}, layer_3::@NamedTuple{layer_1::@NamedTuple{}, layer_2::@NamedTuple{}, layer_3::@NamedTuple{}}}, layer_3::@NamedTuple{}, layer_4::@NamedTuple{}, layer_5::@NamedTuple{}, layer_6::@NamedTuple{layer_1::@NamedTuple{}, layer_2::@NamedTuple{}, layer_3::@NamedTuple{}}, layer_7::@NamedTuple{}, layer_8::@NamedTuple{}, layer_9::@NamedTuple{}, layer_10::@NamedTuple{}}, fixed_depth::Val{0}, init::@NamedTuple{}, solution::DeepEquilibriumSolution, rng::Xoshiro}} at index [3]

In [11]:
#(loss, st), back = Zygote.pullback(ps -> loss_function(model, ps, st, (x_dev, y_dev)), ps)

In [12]:
#loss

In [13]:
#grads = back((one(loss), nothing))